#Import Libraries

In [1]:
# Suppress all warnings
import warnings
warnings.filterwarnings('ignore')

import torch
import pandas as pd
import numpy as np
from datasets import load_dataset
from transformers import BertTokenizer, BertForSequenceClassification
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset
from torch.optim import AdamW
from tqdm import tqdm

# Suppress transformers warnings
import logging
logging.getLogger("transformers").setLevel(logging.ERROR)

# Check GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🚀 GPU Status: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"📊 GPU Model: {torch.cuda.get_device_name(0)}")
    print(f"💾 GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"✅ Using device: {device}\n")

🚀 GPU Status: True
📊 GPU Model: Tesla T4
💾 GPU Memory: 15.6 GB
✅ Using device: cuda



#Load REAL Human vs AI Datasets

In [10]:
print("📚 Loading REAL Human vs AI datasets...")
print("="*50)

# Load human-written texts (XSUM news articles - long form)
print("  → Loading human-written news articles...")
human_dataset = load_dataset("xsum", split="train").select(range(2000))
human_texts = [item["document"][:512] for item in human_dataset]

# Load AI-generated texts (CNN summaries as proxy, but we'll use alternative)
# Better: Use GPT-2 generated texts from dataset
print("  → Loading AI-generated texts...")
try:
    # Try to load ELI5 dataset with generated answers
    ai_dataset = load_dataset("eli5", split="train_eli5").select(range(2000))
    ai_texts = [item["answers"]["text"][0][:512] if len(item["answers"]["text"]) > 0 else "" for item in ai_dataset]
    ai_texts = [text for text in ai_texts if len(text) > 50][:2000]
except:
    # Fallback: Use CNN summaries as AI proxy
    print("  → Fallback: Using CNN summaries as AI texts...")
    ai_dataset = load_dataset("cnn_dailymail", "3.0.0", split="train").select(range(2000))
    ai_texts = [item["highlights"][:512] for item in ai_dataset]

print(f"\n✅ Loaded {len(human_texts)} human texts")
print(f"✅ Loaded {len(ai_texts)} AI texts")
print(f"\n📝 Human sample:\n{human_texts[0][:200]}...")
print(f"\n🤖 AI sample:\n{ai_texts[0][:200]}...")

📚 Loading REAL Human vs AI datasets...
  → Loading human-written news articles...
  → Loading AI-generated texts...


README.md: 0.00B [00:00, ?B/s]

eli5.py: 0.00B [00:00, ?B/s]

  → Fallback: Using CNN summaries as AI texts...

✅ Loaded 2000 human texts
✅ Loaded 2000 AI texts

📝 Human sample:
The full cost of damage in Newton Stewart, one of the areas worst affected, is still being assessed.
Repair work is ongoing in Hawick and many roads in Peeblesshire remain badly affected by standing w...

🤖 AI sample:
Harry Potter star Daniel Radcliffe gets £20M fortune as he turns 18 Monday .
Young actor says he has no plans to fritter his cash away .
Radcliffe's earnings from first five Potter films have been hel...


#Create Balanced Dataset

In [11]:
print("\n📊 Creating balanced dataset...")
print("="*50)

min_samples = min(len(human_texts), len(ai_texts))
print(f"  → Using {min_samples} samples from each class")

data = []
for i in range(min_samples):
    data.append([human_texts[i], 0])  # 0 = Human
    data.append([ai_texts[i], 1])     # 1 = AI

df = pd.DataFrame(data, columns=["text", "label"])
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

print(f"✅ Total samples: {len(df)}")
print(f"  → Human (label 0): {(df['label']==0).sum()}")
print(f"  → AI (label 1): {(df['label']==1).sum()}")
print(f"\n📊 First 5 samples:")
print(df.head())

# Show text length statistics
human_lengths = df[df['label']==0]['text'].str.len()
ai_lengths = df[df['label']==1]['text'].str.len()
print(f"\n📏 Text length stats:")
print(f"  → Human avg length: {human_lengths.mean():.0f} chars")
print(f"  → AI avg length: {ai_lengths.mean():.0f} chars")


📊 Creating balanced dataset...
  → Using 2000 samples from each class
✅ Total samples: 4000
  → Human (label 0): 2000
  → AI (label 1): 2000

📊 First 5 samples:
                                                text  label
0  Several Marines are the focus of 2004 Falluja ...      1
1  Afghan President Hamid Karzai says his country...      1
2  Chester Stiles' ex-girlfriend says she's "disg...      1
3  NIH: Domestic violence is the most common caus...      1
4  Fishing boat operator endures Hurricane Dolly ...      1

📏 Text length stats:
  → Human avg length: 498 chars
  → AI avg length: 255 chars


#Tokenizing texts

In [12]:
print("\n🔤 Tokenizing texts...")
print("="*50)

# Split data
train_texts, val_texts, train_labels, val_labels = train_test_split(
    df["text"].values,
    df["label"].values,
    test_size=0.2,
    random_state=42,
    stratify=df["label"].values
)

print(f"Training samples: {len(train_texts)}")
print(f"Validation samples: {len(val_texts)}")

# Load tokenizer
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

# Tokenize
print("\n  → Tokenizing training set...")
train_encodings = tokenizer(
    list(train_texts),
    truncation=True,
    padding=True,
    max_length=256,
    return_tensors='pt'
)

print("  → Tokenizing validation set...")
val_encodings = tokenizer(
    list(val_texts),
    truncation=True,
    padding=True,
    max_length=256,
    return_tensors='pt'
)

print(f"\n✅ Training tensor shape: {train_encodings['input_ids'].shape}")
print(f"✅ Validation tensor shape: {val_encodings['input_ids'].shape}")


🔤 Tokenizing texts...
Training samples: 3200
Validation samples: 800

  → Tokenizing training set...
  → Tokenizing validation set...

✅ Training tensor shape: torch.Size([3200, 234])
✅ Validation tensor shape: torch.Size([800, 221])


#Create DataLoaders

In [13]:
print("\n📦 Creating DataLoaders...")
print("="*50)

# Create TensorDatasets
train_dataset = TensorDataset(
    train_encodings['input_ids'],
    train_encodings['attention_mask'],
    torch.tensor(train_labels)
)

val_dataset = TensorDataset(
    val_encodings['input_ids'],
    val_encodings['attention_mask'],
    torch.tensor(val_labels)
)

# Create DataLoaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)

print(f"✅ Training batches: {len(train_loader)}")
print(f"✅ Validation batches: {len(val_loader)}")
print(f"✅ Batch size: 32 for training, 64 for validation")


📦 Creating DataLoaders...
✅ Training batches: 100
✅ Validation batches: 13
✅ Batch size: 32 for training, 64 for validation


#Load BERT Model

In [14]:
print("\n🧠 Loading BERT model...")
print("="*50)

model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2
)
model.to(device)

# Optimizer
optimizer = AdamW(model.parameters(), lr=2e-5)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"✅ Model loaded on {device}")
print(f"📊 Total parameters: {total_params:,}")
print(f"📊 Trainable parameters: {trainable_params:,}")
print(f"📊 Learning rate: 2e-5")


🧠 Loading BERT model...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

✅ Model loaded on cuda
📊 Total parameters: 109,483,778
📊 Trainable parameters: 109,483,778
📊 Learning rate: 2e-5


#Training with History Tracking

In [15]:
print("\n🏋️ Starting training...")
print("="*50)

epochs = 3
best_accuracy = 0
training_history = []

for epoch in range(epochs):
    # Training phase
    model.train()
    total_loss = 0
    train_progress = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [Train]")

    for batch in train_progress:
        # Move batch to GPU
        input_ids = batch[0].to(device)
        attention_mask = batch[1].to(device)
        labels = batch[2].to(device)

        # Forward pass
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss
        total_loss += loss.item()

        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # Update progress bar
        train_progress.set_postfix({'loss': f'{loss.item():.4f}'})

    avg_train_loss = total_loss / len(train_loader)

    # Validation phase
    model.eval()
    correct = 0
    total = 0
    all_predictions = []
    all_labels = []

    val_progress = tqdm(val_loader, desc=f"Epoch {epoch+1}/{epochs} [Val]")

    with torch.no_grad():
        for batch in val_progress:
            input_ids = batch[0].to(device)
            attention_mask = batch[1].to(device)
            labels = batch[2].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

            predictions = torch.argmax(outputs.logits, dim=1)
            correct += (predictions == labels).sum().item()
            total += labels.size(0)

            all_predictions.extend(predictions.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

            val_progress.set_postfix({'acc': f'{correct/total:.4f}'})

    accuracy = correct / total

    # Store history
    training_history.append({
        'epoch': epoch + 1,
        'train_loss': avg_train_loss,
        'val_accuracy': accuracy
    })

    print(f"\n📊 Epoch {epoch+1} Results:")
    print(f"  → Train Loss: {avg_train_loss:.4f}")
    print(f"  → Validation Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")

    # Save best model
    if accuracy > best_accuracy:
        best_accuracy = accuracy
        model.save_pretrained("best_human_ai_model")
        tokenizer.save_pretrained("best_human_ai_model")
        print(f"  ✅ Saved best model with accuracy: {accuracy:.4f}")

    print("-"*50)

print(f"\n🎉 Training Complete!")
print(f"🏆 Best Validation Accuracy: {best_accuracy:.4f} ({best_accuracy*100:.2f}%)")

# Display training history
print("\n📈 Training History:")
history_df = pd.DataFrame(training_history)
print(history_df.to_string(index=False))


🏋️ Starting training...


Epoch 1/3 [Val]: 100%|██████████| 13/13 [00:11<00:00,  1.15it/s, acc=0.9975]


📊 Epoch 1 Results:
  → Train Loss: 0.1047
  → Validation Accuracy: 0.9975 (99.75%)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  ✅ Saved best model with accuracy: 0.9975
--------------------------------------------------


Epoch 2/3 [Val]: 100%|██████████| 13/13 [00:11<00:00,  1.15it/s, acc=0.9962]



📊 Epoch 2 Results:
  → Train Loss: 0.0127
  → Validation Accuracy: 0.9962 (99.62%)
--------------------------------------------------


Epoch 3/3 [Val]: 100%|██████████| 13/13 [00:11<00:00,  1.15it/s, acc=0.9962]


📊 Epoch 3 Results:
  → Train Loss: 0.0038
  → Validation Accuracy: 0.9962 (99.62%)
--------------------------------------------------

🎉 Training Complete!
🏆 Best Validation Accuracy: 0.9975 (99.75%)

📈 Training History:
 epoch  train_loss  val_accuracy
     1    0.104668       0.99750
     2    0.012651       0.99625
     3    0.003764       0.99625


#Advanced Testing with Visual Indicators

In [16]:
def predict_text(text):
    """Predict if text is human or AI generated"""
    model.eval()
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=256,
        padding=True
    ).to(device)

    with torch.no_grad():
        outputs = model(**inputs)
        probs = torch.softmax(outputs.logits, dim=1)
        prediction = torch.argmax(probs, dim=1).item()
        confidence = probs[0][prediction].item()

    label = "🤖 AI-GENERATED" if prediction == 1 else "👤 HUMAN-WRITTEN"
    return label, confidence, probs[0][0].item(), probs[0][1].item()

print("\n🔍 Testing Model on Various Texts:")
print("="*60)

test_cases = [
    ("Casual conversation", "Hey! How are you doing today? Let's catch up over coffee sometime."),
    ("Technical content", "The implementation of transformer architectures has revolutionized natural language processing and machine learning tasks."),
    ("Personal story", "I went to the grocery store yesterday and bought some apples and oranges. The cashier was very friendly."),
    ("Scientific", "Based on quantum mechanical principles, particles exhibit wave-particle duality and quantum entanglement properties."),
    ("Email", "Dear Professor Smith, I am writing to inquire about the research position in your laboratory. I have attached my CV for your review."),
    ("News headline", "Breaking: Scientists discover new species of deep-sea creature in the Pacific Ocean."),
]

for description, text in test_cases:
    label, conf, human_prob, ai_prob = predict_text(text)
    print(f"\n📝 {description}:")
    print(f"   Text: {text[:80]}...")
    print(f"   → {label}")
    print(f"   → Confidence: {conf:.2%}")
    print(f"   → Human: {human_prob:.1%} | AI: {ai_prob:.1%}")

    # Visual indicator
    bar_length = 20
    human_bar = int(human_prob * bar_length)
    ai_bar = int(ai_prob * bar_length)
    print(f"   → 👤 {'█' * human_bar}{'░' * (bar_length - human_bar)}")
    print(f"   → 🤖 {'█' * ai_bar}{'░' * (bar_length - ai_bar)}")


🔍 Testing Model on Various Texts:

📝 Casual conversation:
   Text: Hey! How are you doing today? Let's catch up over coffee sometime....
   → 👤 HUMAN-WRITTEN
   → Confidence: 58.42%
   → Human: 58.4% | AI: 41.6%
   → 👤 ███████████░░░░░░░░░
   → 🤖 ████████░░░░░░░░░░░░

📝 Technical content:
   Text: The implementation of transformer architectures has revolutionized natural langu...
   → 🤖 AI-GENERATED
   → Confidence: 95.55%
   → Human: 4.5% | AI: 95.5%
   → 👤 ░░░░░░░░░░░░░░░░░░░░
   → 🤖 ███████████████████░

📝 Personal story:
   Text: I went to the grocery store yesterday and bought some apples and oranges. The ca...
   → 🤖 AI-GENERATED
   → Confidence: 77.24%
   → Human: 22.8% | AI: 77.2%
   → 👤 ████░░░░░░░░░░░░░░░░
   → 🤖 ███████████████░░░░░

📝 Scientific:
   Text: Based on quantum mechanical principles, particles exhibit wave-particle duality ...
   → 🤖 AI-GENERATED
   → Confidence: 99.81%
   → Human: 0.2% | AI: 99.8%
   → 👤 ░░░░░░░░░░░░░░░░░░░░
   → 🤖 ███████████████████░

📝 Email

#Error Analysis & Metrics

In [17]:
from sklearn.metrics import confusion_matrix, classification_report
import matplotlib.pyplot as plt

print("\n📊 Error Analysis")
print("="*50)

# Get all predictions on validation set
model.eval()
all_preds = []
all_true = []

with torch.no_grad():
    for batch in val_loader:
        input_ids = batch[0].to(device)
        attention_mask = batch[1].to(device)
        labels = batch[2].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        predictions = torch.argmax(outputs.logits, dim=1)

        all_preds.extend(predictions.cpu().numpy())
        all_true.extend(labels.cpu().numpy())

# Calculate metrics
cm = confusion_matrix(all_true, all_preds)
print("\nConfusion Matrix:")
print(f"              Predicted")
print(f"              Human   AI")
print(f"Actual Human  {cm[0,0]:5d}  {cm[0,1]:5d}")
print(f"       AI     {cm[1,0]:5d}  {cm[1,1]:5d}")

# Classification report
print("\nClassification Report:")
print(classification_report(all_true, all_preds, target_names=['Human', 'AI']))

# Calculate accuracy per class
human_accuracy = cm[0,0] / (cm[0,0] + cm[0,1]) if (cm[0,0] + cm[0,1]) > 0 else 0
ai_accuracy = cm[1,1] / (cm[1,0] + cm[1,1]) if (cm[1,0] + cm[1,1]) > 0 else 0

print(f"\nPer-class Accuracy:")
print(f"  → Human: {human_accuracy:.2%}")
print(f"  → AI: {ai_accuracy:.2%}")


📊 Error Analysis

Confusion Matrix:
              Predicted
              Human   AI
Actual Human    400      0
       AI         3    397

Classification Report:
              precision    recall  f1-score   support

       Human       0.99      1.00      1.00       400
          AI       1.00      0.99      1.00       400

    accuracy                           1.00       800
   macro avg       1.00      1.00      1.00       800
weighted avg       1.00      1.00      1.00       800


Per-class Accuracy:
  → Human: 100.00%
  → AI: 99.25%


#Save Final Model

In [18]:
# Save your trained model (it's still in memory!)
print("💾 Saving your trained model...")
print("="*50)

# Save model and tokenizer
model.save_pretrained("human_ai_detector_final")
tokenizer.save_pretrained("human_ai_detector_final")

print("✅ Model saved successfully!")
print(f"📁 Location: human_ai_detector_final/")
print("\n📄 Saved files:")
!ls -lh human_ai_detector_final/

💾 Saving your trained model...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Model saved successfully!
📁 Location: human_ai_detector_final/

📄 Saved files:
total 419M
-rw-r--r-- 1 root root  814 Apr 24 08:38 config.json
-rw-r--r-- 1 root root 418M Apr 24 08:38 model.safetensors
-rw-r--r-- 1 root root  322 Apr 24 08:38 tokenizer_config.json
-rw-r--r-- 1 root root 695K Apr 24 08:38 tokenizer.json


#Download Model Files

In [19]:
from google.colab import files

# Download each file one by one
files.download('human_ai_detector_final/config.json')
files.download('human_ai_detector_final/model.safetensors')
files.download('human_ai_detector_final/tokenizer.json')
files.download('human_ai_detector_final/tokenizer_config.json')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>